# Re-run the experiments on Colab

Runs the fixed pipeline end to end: extract frozen-backbone features once, then train and
evaluate with a **subject-disjoint** split. It also re-runs the original frame-level split on
the same features, so the README can report exactly how much the leak inflated accuracy.

**Runtime:** GPU (A100 or L4 recommended). Feature extraction is the slow step; features are
saved to Google Drive so a disconnect doesn't lose them.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

## 1. Install

In [ ]:
%cd /content
!git clone -q https://github.com/Anushkajha25/gait-condition-classifier.git
%cd /content/gait-condition-classifier
!pip install -q -e ".[dev]"

## 2. Unpack CASIA-B
Point `ZIP_PATH` at your copy of the dataset on Drive.

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/CASIA B dataset.zip'
FEATURES_DIR = '/content/drive/MyDrive/gait_features'   # persists across sessions

import os, zipfile, pathlib
if not pathlib.Path('/content/casia').exists():
    zipfile.ZipFile(ZIP_PATH).extractall('/content/casia')

# find the folder that directly contains the subject folders 001/ ... 124/
DATA_ROOT = next(str(p.parent) for p in pathlib.Path('/content/casia').rglob('001')
                 if p.is_dir() and (p.parent / '124').exists())
print('DATA_ROOT =', DATA_ROOT)

## 3. Smoke test (about 2 minutes)
Catches environment problems before the long run.

In [ ]:
!python -m pytest -q
!python scripts/extract_features.py --data-root "$DATA_ROOT" --features-dir /content/smoke_features --limit 512

## 4. Extract features (the long step)
All four backbones in one pass; each image is decoded once.

In [ ]:
!python scripts/extract_features.py --data-root "$DATA_ROOT" --features-dir "$FEATURES_DIR"

## 5. Experiments

In [ ]:
# main result: ResNet101 + EfficientNetB2, subject-disjoint split
!python scripts/run_experiment.py --features-dir "$FEATURES_DIR" --output-dir results/resnet_effnet_subject

# backbone comparison, same split
!python scripts/run_experiment.py --features-dir "$FEATURES_DIR" --backbones vgg16 inceptionv3 --output-dir results/vgg_inception_subject

# the leak, measured: same features, original frame-level split
!python scripts/run_experiment.py --features-dir "$FEATURES_DIR" --split frame --output-dir results/resnet_effnet_frame
!python scripts/run_experiment.py --features-dir "$FEATURES_DIR" --backbones vgg16 inceptionv3 --split frame --output-dir results/vgg_inception_frame

## 6. Compare and save

In [ ]:
import pandas as pd
runs = ['resnet_effnet_subject', 'vgg_inception_subject', 'resnet_effnet_frame', 'vgg_inception_frame']
table = pd.concat([pd.read_csv(f'results/{r}/summary.csv').assign(run=r) for r in runs])
cols = ['run', 'model', 'frame_acc', 'frame_balanced_acc', 'seq_acc', 'majority_baseline',
        'recall_nm', 'recall_bg', 'recall_cl']
display(table[cols].round(4))

# keep everything except the large prediction arrays and model weights
!mkdir -p /content/drive/MyDrive/gait_results
!rsync -a --exclude 'models/' --exclude '*.npy' results/ /content/drive/MyDrive/gait_results/

Then commit `results/<run>/{summary.csv,metrics.json,split.csv,config.yaml,*.png}` to the repo and
fill in the README tables. Don't commit `models/` or `*.npy` (already in `.gitignore`).